# CS781 Phase 7 — Notebook P7_01: Prior Case Retrieval data, checks, lexical baselines, token files

No neural network is trained here. The notebook prepares everything P7_02 needs, once:

| Step | What happens |
|---|---|
| 1 | download `Phase_7_PCR` from Hugging Face; parse queries (id, sentences, relevant ids) and candidate pools. **Test relevance labels are never read** (the field is absent anyway). |
| 2 | checks: duplicate ids/texts, gold ids missing from the pool, query = candidate (self), test queries that also appear in train/dev, pool overlaps |
| 3 | statistics: lengths (sentences, words, our BPE tokens, 512-token chunks), relevant candidates per query, pool sizes, citation masks |
| 4 | **lexical baselines** — our own BM25 on uni-, bi-, tri-grams (sparse matrices, no library ranker) on train and dev with the official metric Micro-F1@7 (+ precision, recall, F1@K for K = 1…20, recall@100/200 = ceiling for re-ranking) |
| 5 | fusion of the n-gram BM25 scores + a **citation-popularity prior** learned from training labels (leave-one-query-out on train, checked on dev) |
| 6 | save: BPE token files of every query and candidate (with sentence boundaries), gold labels, all lexical score matrices (BM25 hard negatives / fusion features for P7_02), a BM25 baseline submission |

**Kaggle settings**: Accelerator **None**, Internet **ON**, secret `HF`. **Input**: the **NB1 output** (tokenizer).
**Output**: `/kaggle/working/cs781_p1/p7/data/…` → attach to P7_02.

**Metric** (spec §13.4): for every query exactly 7 unique candidates from its own split's pool are ranked; over all
queries `P = Σ hits / Σ 7`, `R = Σ hits / Σ |gold|`, `F1 = 2PR / (P + R)` (micro-averaged). The gold set is not cut or
padded to 7, so a query with 15 gold cases can reach at most 7/15 recall.

In [ ]:
!pip -q install tokenizers

In [ ]:
# ============================================================================================
# SHARED SETUP  —  paste this SAME cell at the top of every Phase-1 notebook.
# Works on Kaggle (Save & Commit or interactive), Colab (Drive) and local machines.
# ============================================================================================
import os, sys, json, time, glob, shutil, random, math, re, zlib, hashlib, gc
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")

SESSION_START = time.time()
TIME_BUDGET_H = 11.0   # Kaggle kills a session at 12h and then saves NOTHING -> every long loop stops itself before this

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")
PROJECT   = "cs781_p1"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT, SCRATCH = Path("/content/drive/MyDrive") / PROJECT, Path("/content/scratch")   # Drive = persistent
elif IN_KAGGLE:
    ROOT, SCRATCH = Path("/kaggle/working") / PROJECT, Path("/tmp/cs781_scratch")       # working = saved on commit
else:
    ROOT, SCRATCH = Path("./") / PROJECT, Path("./scratch")
ROOT    = Path(os.environ.get("CS781_ROOT", ROOT))
SCRATCH = Path(os.environ.get("CS781_SCRATCH", SCRATCH))
ROOT.mkdir(parents=True, exist_ok=True); SCRATCH.mkdir(parents=True, exist_ok=True)

ENV = "colab" if IN_COLAB else "kaggle" if IN_KAGGLE else "local"
print(f"env={ENV}  ROOT={ROOT}  SCRATCH={SCRATCH}")
for p in {ROOT, SCRATCH}:
    du = shutil.disk_usage(p); print(f"  disk @ {p}: free {du.free/1e9:.1f} GB / total {du.total/1e9:.1f} GB")


# ---------------------------------------------------------------- small utilities
def time_left_h():
    return TIME_BUDGET_H - (time.time() - SESSION_START) / 3600

def save_json(obj, path):
    """Atomic write: a crash mid-write can never leave a half-written (corrupt) state file."""
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=1)
    os.replace(tmp, path)

def load_json(path, default=None):
    path = Path(path)
    if not path.exists(): return default
    with open(path, encoding="utf-8") as f: return json.load(f)

def restore_from_kaggle_inputs(subdirs):
    """Kaggle 'Save & Commit' starts every run with an EMPTY /kaggle/working.
    To resume, attach the output of the previous version (Add Input -> Your Work -> Notebook output).
    This copies <PROJECT>/<subdir> from any attached input into ROOT (never overwriting newer local files)."""
    if not IN_KAGGLE: return
    for sub in subdirs:
        for cand in sorted(glob.glob(f"/kaggle/input/**/{PROJECT}/{sub}", recursive=True)):
            n = 0
            for src in Path(cand).rglob("*"):
                if src.is_file():
                    dst = ROOT / sub / src.relative_to(cand)
                    if not dst.exists():
                        dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, dst); n += 1
            print(f"restored {n} files: {cand} -> {ROOT/sub}")

def find_artifact(rel):
    """Look for ROOT/rel first, then inside any attached Kaggle input."""
    p = ROOT / rel
    if p.exists(): return p
    if IN_KAGGLE:
        hits = sorted(glob.glob(f"/kaggle/input/**/{PROJECT}/{rel}", recursive=True))
        if hits: return Path(hits[0])
    return None

def get_hf_token():
    if os.environ.get("HF_TOKEN"): return os.environ["HF_TOKEN"]
    try:
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("HF")
        if IN_COLAB:
            from google.colab import userdata
            return userdata.get("HF")
    except Exception as e:
        print("Could not read secret 'HF':", e)
    return None

HF_TOKEN = get_hf_token()
if HF_TOKEN: os.environ["HF_TOKEN"] = HF_TOKEN
print("HF token found:", bool(HF_TOKEN))

# Optional: a PRIVATE HF dataset repo used only to move YOUR OWN artifacts between accounts/sessions.
HF_ARTIFACT_REPO = None   # e.g. "your-hf-username/cs781-p1-artifacts"

def hf_push(subdir):
    if not HF_ARTIFACT_REPO: print("HF_ARTIFACT_REPO not set; skip push"); return
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo(HF_ARTIFACT_REPO, repo_type="dataset", private=True, exist_ok=True)
    api.upload_folder(folder_path=str(ROOT / subdir), path_in_repo=subdir,
                      repo_id=HF_ARTIFACT_REPO, repo_type="dataset")
    print("pushed", subdir)

def hf_pull(subdir):
    if not HF_ARTIFACT_REPO: return
    from huggingface_hub import snapshot_download
    snapshot_download(HF_ARTIFACT_REPO, repo_type="dataset", allow_patterns=[f"{subdir}/*", f"{subdir}/**"],
                      local_dir=str(ROOT), token=HF_TOKEN)
    print("pulled", subdir)


# ---------------------------------------------------------------- data utilities
# These MUST be identical in every notebook (the train/val split and cleaning depend on them).
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
SOURCE_NAMES = ["acts", "high_courts", "supreme_court", "tribunals"]   # processing order (matters for dedup)
VAL_PER_10K  = 100            # 1% of documents of every source are held out (document-level split)
PROMPT_WORDS, CONT_WORDS = 512, 128

def extract_text(rec):
    if rec is None: return ""
    if isinstance(rec, str): return rec
    if isinstance(rec, list): return "\n".join(extract_text(r) for r in rec)
    if isinstance(rec, dict):
        for k in ("text", "content", "document", "judgment", "judgement", "body"):
            if k in rec: return extract_text(rec[k])
    return ""

def iter_docs(path):
    """Stream documents from a (possibly multi-GB) JSON file WITHOUT loading it all into RAM.
    Handles: [ {...}, ... ]   |   { id: {...}, ... }   |   JSON-lines."""
    import ijson
    with open(path, "rb") as f:
        head = f.read(1 << 16).lstrip()
        f.seek(0)
        if head[:1] == b"[":
            for rec in ijson.items(f, "item", use_float=True):
                yield extract_text(rec)
            return
        first_line = f.readline(50_000_000); second = f.read(1 << 10).lstrip()[:1]
        f.seek(0)
        is_jsonl = False
        if first_line.endswith(b"\n") and second == b"{":
            try: json.loads(first_line); is_jsonl = True
            except Exception: pass
        if is_jsonl:
            for line in f:
                if line.strip(): yield extract_text(json.loads(line))
        else:
            for _, rec in ijson.kvitems(f, "", use_float=True):
                yield extract_text(rec)

_ZW = re.compile(r"[​‌‍﻿­]")
_WS_ALL = re.compile(r"\s+")
_HSPACE = re.compile(r"[^\S\n]+")
_MANY_NL = re.compile(r"\n\s*\n\s*\n+")

def normalize(text, mode):
    """'collapse_all'  -> every whitespace run becomes one space (text looks like ' '.join(text.split()))
       'keep_newlines' -> collapse spaces/tabs, keep at most one blank line"""
    text = _ZW.sub("", text)
    if mode == "collapse_all":
        return _WS_ALL.sub(" ", text).strip()
    text = _HSPACE.sub(" ", text.replace("\r\n", "\n").replace("\r", "\n"))
    text = "\n".join(l.strip() for l in text.split("\n"))
    return _MANY_NL.sub("\n\n", text).strip()

def hash64(text):
    return int.from_bytes(hashlib.blake2b(text.encode("utf-8", "ignore"), digest_size=8).digest(), "little")

def doc_split(source, idx):
    return "val" if zlib.crc32(f"{source}:{idx}".encode()) % 10000 < VAL_PER_10K else "train"

def list_phase1_files(local_raw_dir=None):
    """Return {'train': {source: repo_path}, 'test': repo_path, 'sizes': {repo_path: bytes}}."""
    if local_raw_dir:
        files = [str(Path(p).relative_to(local_raw_dir)) for p in glob.glob(f"{local_raw_dir}/**/*.json*", recursive=True)]
        sizes = {f: os.path.getsize(Path(local_raw_dir) / f) for f in files}
    else:
        from huggingface_hub import HfApi
        api = HfApi(token=HF_TOKEN)
        try:
            entries = list(api.list_repo_tree(HF_DATASET_REPO, repo_type="dataset",
                                              path_in_repo="Phase_1_Pretraining", recursive=True))
            files = [e.path for e in entries if getattr(e, "size", None) is not None]
            sizes = {e.path: e.size for e in entries if getattr(e, "size", None) is not None}
        except Exception as e:
            print("list_repo_tree failed (", e, ") -> falling back to list_repo_files")
            files = [f for f in api.list_repo_files(HF_DATASET_REPO, repo_type="dataset") if f.startswith("Phase_1")]
            sizes = {}
    out = {"train": {}, "test": None, "sizes": sizes}
    for f in files:
        name = Path(f).name.lower()
        if not name.endswith((".json", ".jsonl")): continue
        if "test" in name: out["test"] = f
        for s in SOURCE_NAMES:
            if name.startswith("train") and s in name: out["train"][s] = f
    return out

def get_raw_file(repo_path, local_raw_dir=None):
    if local_raw_dir:
        return Path(local_raw_dir) / repo_path
    from huggingface_hub import hf_hub_download
    return Path(hf_hub_download(HF_DATASET_REPO, repo_path, repo_type="dataset", token=HF_TOKEN,
                                local_dir=str(SCRATCH / "raw")))

def drop_raw_file(path, local_raw_dir=None):
    if local_raw_dir: return          # never delete user-provided files
    try: os.remove(path); print("deleted raw", path)
    except OSError: pass

## 0. Configuration and download

In [ ]:
import pandas as pd, scipy.sparse as sp
from sklearn.feature_extraction.text import HashingVectorizer, ENGLISH_STOP_WORDS
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
try: display
except NameError: display = print

P7_DATA = ROOT / "p7" / "data"; P7_DATA.mkdir(parents=True, exist_ok=True)
P7_BASE = ROOT / "p7" / "baselines"; P7_BASE.mkdir(parents=True, exist_ok=True)
P7_REPO_DIR = "Phase_7_PCR"
P7_FILES = {"train": ("Train/train.json", "Train/train_candidates.json"),
            "dev":   ("Dev/dev.json", "Dev/dev_candidates.json"),
            "test":  ("Test/test.json", "Test/test_candidates.json")}
SPLITS = ("train", "dev", "test")
LABELLED = ("train", "dev")             # test labels are private: never looked for, never read
TOKENIZER = "bpe32768"
CHUNK_TOK = 512                         # chunk length P7_02 will most likely use (only for statistics here)
K_EVAL = 7
TOPN_SAVE = 200                         # BM25 top-N candidates saved per query (hard negatives / re-ranking)
P7_LOCAL = os.environ.get("CS781_P7_LOCAL")

def get_p7_file(rel):
    if P7_LOCAL: return Path(P7_LOCAL) / rel
    from huggingface_hub import hf_hub_download
    return Path(hf_hub_download(HF_DATASET_REPO, f"{P7_REPO_DIR}/{rel}", repo_type="dataset", token=HF_TOKEN, local_dir=str(SCRATCH / "p7raw")))

PATHS = {s: tuple(get_p7_file(r) for r in rels) for s, rels in P7_FILES.items()}
for s, (q, c) in PATHS.items(): print(f"{s}: {q.name} {os.path.getsize(q)/1e6:.1f} MB | {c.name} {os.path.getsize(c)/1e6:.1f} MB")

## 1. Parse
Field names are matched flexibly (`id`/`text`/`relevant_candidates` per the spec, plus common alternatives); a file may
be a list of records or a dict `{id: record}` / `{id: [sentences]}`. Ids are kept as strings exactly as the **pool**
writes them (the submission must use the pool's spelling); matching between label ids and pool ids ignores leading
zeros, in case one side is zero-padded and the other is not.

In [ ]:
ID_KEYS = ("id", "case_id", "doc_id", "query_id", "candidate_id", "name")
TEXT_KEYS = ("text", "sentences", "document", "judgment", "doc", "content")
REL_KEYS = ("relevant_candidates", "relevant", "relevant_ids", "positives", "labels", "citations")

def first_key(r, keys): return next((k for k in keys if k in r), None)
def canon(x):
    s = str(x).strip()
    return (s.lstrip("0") or "0") if s.isdigit() else s
def as_sents(x):
    if x is None: return []
    if isinstance(x, str): return [x]
    out = []
    for s in x: out.extend(as_sents(s) if isinstance(s, list) else [str(s)])
    return out

def parse_file(path, with_labels):
    with open(path, encoding="utf-8") as f: obj = json.load(f)
    if isinstance(obj, dict) and len(obj) <= 3 and any(isinstance(v, list) and v and isinstance(v[0], dict) for v in obj.values()):
        obj = next(v for v in obj.values() if isinstance(v, list) and v and isinstance(v[0], dict))   # {"data": [...]} wrapper
    recs = list(obj.items()) if isinstance(obj, dict) else [(None, r) for r in obj]
    ids, sents, rels = [], [], []
    for k, r in recs:
        if not isinstance(r, dict): r = {"text": r}
        ik, tk = first_key(r, ID_KEYS), first_key(r, TEXT_KEYS)
        ids.append(str(r[ik]).strip() if ik else str(k).strip())
        sents.append([normalize(s, "collapse_all") for s in as_sents(r.get(tk)) if s and str(s).strip()])
        if with_labels:
            rk = first_key(r, REL_KEYS)
            rels.append([str(x).strip() for x in (r.get(rk) or [])])
    sample = recs[0][1] if recs else None
    keys = list(sample)[:8] if isinstance(sample, dict) else type(sample).__name__
    del obj; gc.collect()
    return ids, sents, (rels if with_labels else None), keys

Q, C = {}, {}
for s in SPLITS:
    qp, cp = PATHS[s]
    ids, sents, rels, keys = parse_file(qp, with_labels=(s in LABELLED))
    Q[s] = {"ids": ids, "sents": sents, "rels": rels}
    print(f"{s} queries: {len(ids)} records, keys {keys}")
    ids, sents, _, keys = parse_file(cp, with_labels=False)
    C[s] = {"ids": ids, "sents": sents}
    print(f"{s} candidates: {len(ids)} records, keys {keys}")
print("\nfirst train query:", Q["train"]["ids"][0], "|", len(Q["train"]["sents"][0]), "sentences |", " ".join(Q["train"]["sents"][0][:3])[:400])
print("its relevant candidates:", Q["train"]["rels"][0][:10])
print("first train candidate:", C["train"]["ids"][0], "|", " ".join(C["train"]["sents"][0][:2])[:300])

## 2. Checks
* duplicate ids / identical texts inside a pool;
* gold ids that are **not** in the split's pool (cannot be retrieved → lower the reachable recall);
* a query that is itself in its pool (it must never be returned for itself → excluded when ranking);
* test queries whose id or text also appears among train/dev queries (their labels would leak — we would **not** use
  them, only report);
* pool overlap between splits, and how many dev gold cases were ever cited by a training query (is a popularity prior
  useful?).

In [ ]:
def text_key(sents): return hash64(" ".join(sents)[:30000])
CHK = {}
for s in SPLITS:
    cids = C[s]["ids"]; ccanon = [canon(x) for x in cids]
    dup_id = len(cids) - len(set(ccanon))
    th = Counter(text_key(x) for x in C[s]["sents"]); dup_txt = sum(v - 1 for v in th.values() if v > 1)
    empty_c = sum(not x for x in C[s]["sents"]); empty_q = sum(not x for x in Q[s]["sents"])
    pool = set(ccanon); self_in = sum(canon(q) in pool for q in Q[s]["ids"])
    row = {"queries": len(Q[s]["ids"]), "pool": len(cids), "dup pool ids": dup_id, "dup pool texts": dup_txt,
           "empty queries": empty_q, "empty candidates": empty_c, "queries inside own pool": self_in,
           "dup query ids": len(Q[s]["ids"]) - len(set(map(canon, Q[s]["ids"])))}
    if Q[s]["rels"] is not None:
        g = [[canon(x) for x in r] for r in Q[s]["rels"]]
        n_gold = sum(len(r) for r in g); missing = sum(x not in pool for r in g for x in r)
        row.update({"gold total": n_gold, "gold missing from pool": missing, "queries with no gold": sum(not r for r in g),
                    "gold = query itself": sum(canon(q) in set(r) for q, r in zip(Q[s]["ids"], g))})
    CHK[s] = row
display(pd.DataFrame(CHK).T)

lab_q = {canon(q) for s in LABELLED for q in Q[s]["ids"]}
lab_qtext = {text_key(x) for s in LABELLED for x in Q[s]["sents"]}
LEAK = {"test query ids also in train/dev queries": sum(canon(q) in lab_q for q in Q["test"]["ids"]),
        "test query texts also in train/dev queries": sum(text_key(x) in lab_qtext for x in Q["test"]["sents"]),
        "dev query ids also in train queries": sum(canon(q) in {canon(x) for x in Q["train"]["ids"]} for q in Q["dev"]["ids"])}
pools = {s: {canon(x) for x in C[s]["ids"]} for s in SPLITS}
for a, b in (("train", "dev"), ("train", "test"), ("dev", "test")):
    LEAK[f"pool overlap {a}∩{b}"] = len(pools[a] & pools[b])
cited_train = Counter(canon(x) for r in Q["train"]["rels"] for x in r)
dev_gold = [canon(x) for r in Q["dev"]["rels"] for x in r]
LEAK["dev gold cases cited by ≥1 train query"] = f"{np.mean([x in cited_train for x in dev_gold]):.1%}" if dev_gold else "n/a"
dev_gold_set = set(dev_gold)
LEAK["test-pool cases cited by ≥1 train/dev query"] = f"{np.mean([x in cited_train or x in dev_gold_set for x in pools['test']]):.1%}"
for k, v in LEAK.items(): print(f"  {k}: {v}")
if LEAK["test query ids also in train/dev queries"] or LEAK["test query texts also in train/dev queries"]:
    print("  !! some test queries also occur in train/dev with labels — we do NOT use those labels for test predictions; tell the TAs.")

## 3. Statistics
Words are whitespace tokens; BPE tokens are counted with our Phase-1 tokenizer (step 6). Citation masks: IL-PCR
replaces the citations inside a query by a placeholder — we count the most common bracketed / upper-case placeholders.

In [ ]:
def words(sents): return sum(len(x.split()) for x in sents)
rows = []
for s in SPLITS:
    for role, D in (("queries", Q[s]), ("candidates", C[s])):
        w = np.array([words(x) for x in D["sents"]]); n = np.array([len(x) for x in D["sents"]])
        rows.append({"split": s, "role": role, "docs": len(w), "words median": np.median(w), "words mean": w.mean(),
                     "words p90": np.percentile(w, 90), "words max": w.max(), "sentences median": np.median(n)})
display(pd.DataFrame(rows).round(0))
for s in LABELLED:
    g = np.array([len(r) for r in Q[s]["rels"]])
    print(f"{s}: relevant per query — mean {g.mean():.2f}, median {np.median(g):.0f}, max {g.max()}, "
          f"≤7: {np.mean(g <= 7):.1%} | best possible Micro-F1@7 (oracle) {2*np.minimum(g, 7).sum()/(7*len(g)+g.sum()):.3f}")
mask_pat = re.compile(r"<[^<>\s]{2,30}>|\[[A-Z_ ]{3,30}\]|\b[A-Z_]{6,}\b")
mc = Counter(m for x in Q["train"]["sents"][:300] for sent in x for m in mask_pat.findall(sent))
print("most common placeholders in train queries:", mc.most_common(8))
cpop = np.array(sorted(cited_train.values(), reverse=True))
print(f"train citations: {len(cited_train)} distinct cited cases; top-10 most cited are cited {cpop[:10].tolist()} times; "
      f"{np.mean(cpop == 1):.1%} cited only once")

## 4. Lexical baselines: BM25 on n-grams (own implementation)
Text → lower case → alphanumeric tokens (numbers kept: section numbers and years matter in law) → English stop words
and the citation placeholder removed → n-grams. Every n-gram is hashed into 2^24 buckets (`HashingVectorizer`, so no
vocabulary has to be held in memory; collisions are negligible at this size).

**BM25** (Robertson & Zaragoza 2009), pool statistics per split:
`score(q, d) = Σ_t w_q(t) · idf(t) · tf(t,d)·(k1+1) / (tf(t,d) + k1·(1 − b + b·|d|/avgdl))`, `idf = ln(1 + (N − df + 0.5)/(df + 0.5))`,
with `w_q` = the term's count in the query (`raw`) or 1 (`bin`). Uni-/bi-/tri-gram BM25 follows Joshi et al. 2023
(U-CREAT, IL-PCR), where n-gram BM25 is a strong baseline. The whole score matrix is one sparse product `Q · Wᵀ`.

In [ ]:
STOP = frozenset(ENGLISH_STOP_WORDS) | {"citation", "citations"}
TOKRE = re.compile(r"[a-z0-9]+")
def clean(sents):
    toks = TOKRE.findall(" ".join(sents).lower())
    return " ".join(t for t in toks if t not in STOP and (len(t) > 1 or t.isdigit()))
CLEAN = {s: {"q": [clean(x) for x in Q[s]["sents"]], "c": [clean(x) for x in C[s]["sents"]]} for s in SPLITS}
print("cleaned example:", CLEAN["train"]["q"][0][:300])

def ngram_analyzer(n):
    def f(text):
        t = text.split()
        return t if n == 1 else [" ".join(t[i:i + n]) for i in range(len(t) - n + 1)]
    return f
HV = {n: HashingVectorizer(analyzer=ngram_analyzer(n), n_features=2 ** 24, alternate_sign=False, norm=None) for n in (1, 2, 3)}

def bm25_matrix(X, k1=1.2, b=0.75):
    X = X.tocsr().astype(np.float32); N = X.shape[0]
    dl = np.asarray(X.sum(1)).ravel(); avg = dl.mean()
    df = np.bincount(X.indices, minlength=X.shape[1]).astype(np.float32)
    idf = np.log1p((N - df + 0.5) / (df + 0.5)).astype(np.float32)
    rows = np.repeat(np.arange(N), np.diff(X.indptr))
    tf = X.data
    X.data = tf * (k1 + 1) / (tf + k1 * (1 - b + b * dl[rows] / avg)) * idf[X.indices]
    return X

def self_mask(s):
    pos = {canon(x): i for i, x in enumerate(C[s]["ids"])}
    return [(qi, pos[canon(q)]) for qi, q in enumerate(Q[s]["ids"]) if canon(q) in pos]

COUNTS = {}                                      # (split, n, "c" | query text key) -> n-gram count matrix, built once
def counts(s, n, qkey="q"):
    for k in ("c", qkey):
        if (s, n, k) not in COUNTS: COUNTS[(s, n, k)] = HV[n].transform(CLEAN[s][k]).tocsr().astype(np.float32)
    return COUNTS[(s, n, "c")], COUNTS[(s, n, qkey)]

def score_matrix(s, n, qmode="raw", k1=1.2, b=0.75, qkey="q"):
    Xc, Xq = counts(s, n, qkey); W = bm25_matrix(Xc.copy(), k1, b); Xq = Xq.copy()
    if qmode == "bin": Xq.data[:] = 1.0
    S = np.zeros((Xq.shape[0], W.shape[0]), np.float32)
    for i in range(0, Xq.shape[0], 256): S[i:i + 256] = (Xq[i:i + 256] @ W.T).toarray()
    for qi, ci in self_mask(s): S[qi, ci] = -np.inf
    return S

GOLD = {}
for s in LABELLED:
    pos = {canon(x): i for i, x in enumerate(C[s]["ids"])}
    GOLD[s] = [sorted({pos[canon(x)] for x in r if canon(x) in pos}) for r in Q[s]["rels"]]
N_GOLD = {s: [len(r) for r in Q[s]["rels"]] for s in LABELLED}     # full gold size (incl. ids missing from the pool)

def topk(S, k):
    k = min(k, S.shape[1]); idx = np.argpartition(-S, k - 1, axis=1)[:, :k]
    o = np.argsort(-np.take_along_axis(S, idx, 1), axis=1)
    return np.take_along_axis(idx, o, 1)

def evaluate(S, s, K=K_EVAL, curve=False):
    order = topk(S, max(K, 200 if curve else K))
    hits_at = np.array([[int(j in set(g)) for j in row] for row, g in zip(order, GOLD[s])])
    ng = sum(N_GOLD[s])
    def at(k):
        h = hits_at[:, :k].sum(); p = h / (k * len(order)); r = h / max(1, ng)
        return {"P": p, "R": r, "F1": 2 * p * r / max(1e-12, p + r)}
    out = {f"{m}@{K}": v for m, v in at(K).items()}
    if curve:
        out["F1 curve"] = [round(at(k)["F1"], 4) for k in range(1, 21)]
        out["R@100"] = hits_at[:, :100].sum() / max(1, ng); out["R@200"] = hits_at[:, :200].sum() / max(1, ng)
    return out

RES, SCORES = {}, {s: {} for s in SPLITS}
t0 = time.time()
for n in (1, 2, 3):
    for qmode in ("raw", "bin"):
        name = f"BM25 {n}-gram {qmode}"
        for s in LABELLED:
            S = score_matrix(s, n, qmode)
            RES[(name, s)] = evaluate(S, s, curve=True)
            if qmode == "raw": SCORES[s][f"bm25_{n}"] = S
        print(f"{name}: train F1@7 {RES[(name, 'train')]['F1@7']:.4f} | dev F1@7 {RES[(name, 'dev')]['F1@7']:.4f}  ({time.time()-t0:.0f}s)")
for n in (1, 2, 3): SCORES["test"][f"bm25_{n}"] = score_matrix("test", n, "raw")
tab = pd.DataFrame({f"{name} | {s}": {k: v for k, v in r.items() if k != "F1 curve"} for (name, s), r in RES.items()}).T
display(tab.round(4))

**k1 / b** of the best n-gram order (grid on train, checked on dev). Long legal documents often prefer a smaller `b`
(less length normalisation) — the grid tells.

In [ ]:
best_n = max((1, 2, 3), key=lambda n: max(RES[(f"BM25 {n}-gram {m}", "train")]["F1@7"] for m in ("raw", "bin")))
grid = []
for qm in ("raw", "bin"):
    for k1 in (0.9, 1.2, 1.6, 2.0):
        for b in (0.5, 0.75, 0.9):
            grid.append({"query": qm, "k1": k1, "b": b, **{f"{s} F1@7": evaluate(score_matrix(s, best_n, qm, k1, b), s)["F1@7"] for s in LABELLED}})
grid = pd.DataFrame(grid).sort_values("train F1@7", ascending=False); display(grid.head(6).round(4))
BEST_QM = grid.iloc[0]["query"]
BEST_KB = (float(grid.iloc[0]["k1"]), float(grid.iloc[0]["b"]))
for s in SPLITS: SCORES[s][f"bm25_{best_n}"] = score_matrix(s, best_n, BEST_QM, *BEST_KB)
print(f"best order n={best_n}, query weights {BEST_QM}, k1={BEST_KB[0]}, b={BEST_KB[1]}")

## 5. Fusion and a citation-popularity prior
* **Fusion**: every score row is z-normalised per query (scores of different n are on different scales), then a
  weighted sum `Σ_n w_n · z_n` (weights from a small grid, chosen on train, reported on dev).
* **Popularity prior**: some cases (landmark judgments) are cited by many queries. `prior(c) = ln(1 + #training
  queries citing c)`; on train it is computed **leave-one-query-out** (a query's own labels never count for itself), on
  dev from all training labels, on test from train + dev labels. Final score `z_fused + β · prior`, β chosen on train.

In [ ]:
def z(S):
    S = S.copy(); fin = np.isfinite(S)
    m = np.where(fin, S, np.nan); mu = np.nanmean(m, 1, keepdims=True); sd = np.nanstd(m, 1, keepdims=True) + 1e-6
    out = (S - mu) / sd; out[~fin] = -np.inf; return out
Z = {s: {k: z(v) for k, v in SCORES[s].items()} for s in SPLITS}

def fused(s, w): return sum(wi * Z[s][f"bm25_{n}"] for n, wi in zip((1, 2, 3), w) if wi)
fgrid = []
for w in [(a, b_, c) for a in (0, .5, 1) for b_ in (0, .5, 1) for c in (0, .5, 1) if a + b_ + c > 0]:
    fgrid.append({"w_uni": w[0], "w_bi": w[1], "w_tri": w[2], **{f"{s} F1@7": evaluate(fused(s, w), s)["F1@7"] for s in LABELLED}})
fgrid = pd.DataFrame(fgrid).sort_values("train F1@7", ascending=False); display(fgrid.head(8).round(4))
W_FUSE = tuple(float(x) for x in fgrid.iloc[0][["w_uni", "w_bi", "w_tri"]])

def prior_vec(s, counts):
    return np.array([np.log1p(counts.get(canon(x), 0)) for x in C[s]["ids"]], np.float32)
cnt_train = Counter(canon(x) for r in Q["train"]["rels"] for x in r)
cnt_traindev = cnt_train + Counter(canon(x) for r in Q["dev"]["rels"] for x in r)
pos_tr = {canon(x): i for i, x in enumerate(C["train"]["ids"])}
def prior_train_loo():
    base = prior_vec("train", cnt_train); P = np.repeat(base[None], len(Q["train"]["ids"]), 0)
    for qi, r in enumerate(Q["train"]["rels"]):
        for x in {canon(y) for y in r}:
            if x in pos_tr: P[qi, pos_tr[x]] = np.log1p(cnt_train[x] - 1)
    return P
PRIOR = {"train": prior_train_loo(), "dev": prior_vec("dev", cnt_train)[None], "test": prior_vec("test", cnt_traindev)[None]}
pg = []
for beta in (0, .1, .2, .3, .5, .75, 1.0, 1.5):
    pg.append({"beta": beta, **{f"{s} F1@7": evaluate(fused(s, W_FUSE) + beta * PRIOR[s], s)["F1@7"] for s in LABELLED}})
pg = pd.DataFrame(pg); display(pg.round(4))
BETA = float(pg.sort_values("train F1@7", ascending=False).iloc[0]["beta"])

SYSTEMS = {f"BM25 {best_n}-gram {BEST_QM} (k1={BEST_KB[0]}, b={BEST_KB[1]})": lambda s: SCORES[s][f"bm25_{best_n}"],
           f"BM25 fusion w={W_FUSE}": lambda s: fused(s, W_FUSE),
           f"BM25 fusion + prior β={BETA}": lambda s: fused(s, W_FUSE) + BETA * PRIOR[s]}
FINAL = {}
for name, f in SYSTEMS.items():
    for s in LABELLED: FINAL[(name, s)] = evaluate(f(s), s, curve=True)
display(pd.DataFrame({f"{n} | {s}": {k: v for k, v in r.items() if k != "F1 curve"} for (n, s), r in FINAL.items()}).T.round(4))
best_sys = max(SYSTEMS, key=lambda n: FINAL[(n, "train")]["F1@7"])
print("F1@K for K = 1…20 on dev (best system):", [float(x) for x in FINAL[(best_sys, "dev")]["F1 curve"]])

## 5b. Lexical++: citation contexts, query weighting, and time
Three legal observations turned into features (no learning except a handful of weights chosen on train):
* **Citation contexts.** In a query every citation is replaced by `<CITATION>`; the sentences around it describe the
  cited case (its facts, the provision, the holding). BM25 with only those sentences (±1 sentence) as the query
  focuses on exactly what the cited cases are about, instead of the whole judgment.
* **Binary query weights.** A phrase that occurs 30 times in a long query should not count 30 times.
* **Time.** A case can only cite *earlier* cases ("prior" case retrieval). Each case's year = the `DATE OF JUDGMENT`
  in its header if present, else the latest year it mentions (ignoring "CITATOR INFO", which lists *later* cases).
  Candidates later than the query (beyond a tolerance) get a penalty γ; tolerance and γ are chosen on train after
  checking how often cited cases actually satisfy the rule.
All 12 features (full query / citation contexts × 1-, 2-, 3-grams × raw / binary) are z-normalised per query and
combined with weights found by coordinate ascent on **train** Micro-F1@7 (dev only reported).

In [ ]:
CIT = "<CITATION>"
CTX = {s: [[i for i, x in enumerate(d) if CIT in x] for d in Q[s]["sents"]] for s in SPLITS}
for s in SPLITS:
    nc = np.array([len(c) for c in CTX[s]])
    print(f"{s}: queries with a <CITATION> sentence {np.mean(nc > 0):.1%} | citation sentences per query: mean {nc.mean():.1f}, median {np.median(nc):.0f}"
          + (f" | gold per query {np.mean(N_GOLD[s]):.1f}" if s in LABELLED else ""))
def ctx_text(sents, ids, w=1):
    keep = sorted({j for i in ids for j in range(max(0, i - w), min(len(sents), i + w + 1))})
    return [sents[j] for j in keep] if keep else sents
for s in SPLITS: CLEAN[s]["qctx"] = [clean(ctx_text(x, c)) for x, c in zip(Q[s]["sents"], CTX[s])]

DATE_RE = re.compile(r"DATE OF JUDGMENT\s*:?\s*\d{1,2}\s*[/.-]\s*\d{1,2}\s*[/.-]\s*((?:18|19|20)\d\d)", re.I)
YEAR_RE = re.compile(r"\b(18[5-9]\d|19\d\d|20[0-2]\d)\b")
def year_of(sents):
    m = DATE_RE.search(" ".join(sents[:20])[:20000])
    if m: return int(m.group(1)), "header"
    ys = [int(y) for x in sents if "citator" not in x.lower() for y in YEAR_RE.findall(x)]
    return (max(ys), "max") if ys else (-1, "none")
YEARS = {s: {r: [year_of(x) for x in D[s]["sents"]] for r, D in (("q", Q), ("c", C))} for s in SPLITS}
for s in SPLITS:
    for r in ("q", "c"):
        src = Counter(t for _, t in YEARS[s][r]); ys = np.array([y for y, _ in YEARS[s][r] if y > 0])
        print(f"  {s} {r}: year from header {src['header']}, from max year {src['max']}, unknown {src['none']} | range {ys.min() if len(ys) else '-'}–{ys.max() if len(ys) else '-'}")
def year_arrays(s):
    return np.array([y for y, _ in YEARS[s]["q"]]), np.array([y for y, _ in YEARS[s]["c"]])
diffs = []
for s in LABELLED:
    qy, cy = year_arrays(s)
    for qi, g in enumerate(GOLD[s]):
        for j in g:
            if qy[qi] > 0 and cy[j] > 0: diffs.append((s, cy[j] - qy[qi]))
for s in LABELLED:
    d = np.array([x for t, x in diffs if t == s])
    print(f"{s}: cited case year − query year over {len(d)} gold pairs: ≤0 {np.mean(d <= 0):.1%}, ≤1 {np.mean(d <= 1):.1%}, ≤2 {np.mean(d <= 2):.1%}, ≤5 {np.mean(d <= 5):.1%}")
def late(s, tol):
    qy, cy = year_arrays(s)
    return (qy[:, None] > 0) & (cy[None, :] > 0) & (cy[None, :] > qy[:, None] + tol)
for s in LABELLED: print(f"{s}: share of all (query, candidate) pairs that are 'later than the query' (tol 0): {late(s, 0).mean():.1%}")

FEAT = {s: {} for s in SPLITS}
t0 = time.time()
for s in SPLITS:
    for qkey, tag in (("q", "full"), ("qctx", "ctx")):
        for n in (1, 2, 3):
            for qm in ("raw", "bin"):
                FEAT[s][f"{tag}{n}_{qm}"] = z(score_matrix(s, n, qm, *BEST_KB, qkey=qkey))
print(f"12 lexical features in {time.time()-t0:.0f}s")
FNAMES = list(FEAT["train"])
display(pd.DataFrame({f: {sp: evaluate(FEAT[sp][f], sp)["F1@7"] for sp in LABELLED} for f in FNAMES}).T.round(4))

def comb(s, w, gamma=0.0, tol=0):
    S = sum(v * FEAT[s][f] for f, v in w.items() if v)
    return S - gamma * late(s, tol) if gamma else S
single = {f: evaluate(FEAT["train"][f], "train")["F1@7"] for f in FNAMES}
W_PP = {f: 0.0 for f in FNAMES}; W_PP[max(single, key=single.get)] = 1.0
cur = single[max(single, key=single.get)]
for rnd in range(3):
    improved = False
    for f in FNAMES:
        for v in (0.0, 0.25, 0.5, 1.0, 1.5, 2.0, 3.0):
            if v == W_PP[f]: continue
            w2 = dict(W_PP, **{f: v})
            if not any(w2.values()): continue
            sc = evaluate(comb("train", w2), "train")["F1@7"]
            if sc > cur + 1e-4: cur, W_PP, improved = sc, w2, True
    print(f"round {rnd + 1}: train F1@7 {cur:.4f} | dev {evaluate(comb('dev', W_PP), 'dev')['F1@7']:.4f} | weights {{{', '.join(f'{k}: {v}' for k, v in W_PP.items() if v)}}}")
    if not improved: break
tg = []
for tol in (0, 1, 2, 5):
    for gamma in (0.0, 0.5, 1.0, 2.0, 5.0, 100.0):
        tg.append({"tol": tol, "gamma": gamma, **{f"{sp} F1@7": evaluate(comb(sp, W_PP, gamma, tol), sp)["F1@7"] for sp in LABELLED}})
tg = pd.DataFrame(tg).sort_values("train F1@7", ascending=False); display(tg.head(8).round(4))
YEAR_TOL, YEAR_GAMMA = int(tg.iloc[0]["tol"]), float(tg.iloc[0]["gamma"])
print(f"time rule: tolerance {YEAR_TOL} years, penalty γ = {YEAR_GAMMA}")

SYSTEMS["lexical++ (no time rule)"] = lambda s: comb(s, W_PP)
SYSTEMS["lexical++"] = lambda s: comb(s, W_PP, YEAR_GAMMA, YEAR_TOL)
for name in ("lexical++ (no time rule)", "lexical++"):
    for sp in LABELLED: FINAL[(name, sp)] = evaluate(SYSTEMS[name](sp), sp, curve=True)
display(pd.DataFrame({f"{n} | {sp}": {k: v for k, v in r.items() if k != "F1 curve"} for (n, sp), r in FINAL.items()}).T.round(4))
best_sys = max(SYSTEMS, key=lambda n: FINAL[(n, "train")]["F1@7"])
print("best lexical system (by train):", best_sys, "| dev F1@7", round(FINAL[(best_sys, "dev")]["F1@7"], 4))

### Error analysis of the best lexical system (dev)
Where do the gold cases end up in the ranking? A gold case ranked 8–50 is a re-ranking opportunity for P7_02; one
ranked > 200 is practically out of reach for a re-ranker that starts from BM25 — the neural model must score the whole
pool to find it.

In [ ]:
Sd = SYSTEMS[best_sys]("dev"); ranks = []
order = np.argsort(-Sd, axis=1)
for qi, g in enumerate(GOLD["dev"]):
    rpos = np.empty(Sd.shape[1], int); rpos[order[qi]] = np.arange(Sd.shape[1])
    ranks += [int(rpos[j]) + 1 for j in g]
ranks = np.array(ranks)
bins = [(1, 7), (8, 20), (21, 50), (51, 100), (101, 200), (201, 10 ** 9)]
print("dev gold cases by rank:", {f"{a}-{b}" if b < 10 ** 9 else f">{a-1}": f"{np.mean((ranks >= a) & (ranks <= b)):.1%}" for a, b in bins})
qf1 = []
for qi, g in enumerate(GOLD["dev"]):
    h = len(set(order[qi, :7]) & set(g)); qf1.append((h, len(Q["dev"]["rels"][qi]), words(Q["dev"]["sents"][qi])))
qf1 = pd.DataFrame(qf1, columns=["hits@7", "gold", "query words"])
print("dev queries with 0 hits in the top 7:", f"{np.mean(qf1['hits@7'] == 0):.1%}")
qf1["length bin"] = pd.qcut(qf1["query words"], 4, duplicates="drop")
display(qf1.groupby("length bin", observed=True)[["hits@7", "gold"]].mean().round(2))

## 6. Save
* **Token files** (Phase-1 BPE): per split and role (`q` = queries, `c` = candidates) one `uint16` stream; every
  sentence is encoded separately (with a leading space, as inside running text) so P7_02 can cut 512-token chunks at
  sentence boundaries. `…_sent_off.npy` = token offset of every sentence (+ end), `…_doc_sent.npy` = index of each
  document's first sentence (+ end).
* **Labels**: `{split}_gold.json` = pool indices of the gold cases (train, dev).
* **Lexical scores**: `{split}_lex.npz` = raw BM25 1/2/3-gram matrices (float16, self = −inf), the per-query top-200
  of the best lexical system, and the popularity prior → hard negatives and fusion features in P7_02.
* **Baseline submission** of the best lexical system (not one of the required submissions: it does not use the
  Phase-1 model; useful only to check that dev and test scores agree).

In [ ]:
from tokenizers import Tokenizer
TOK = Tokenizer.from_file(str(find_artifact(f"tokenizer/{TOKENIZER}/tokenizer.json") or "missing"))
TOKSTATS = {}
def save_tokens(s, role, docs_sents):
    stream, sent_off, doc_sent = [], [0], [0]
    for i in range(0, len(docs_sents), 400):
        batch = docs_sents[i:i + 400]
        flat = [(" " + x) if j else x for d in batch for j, x in enumerate(d)]
        enc = TOK.encode_batch(flat, add_special_tokens=False) if flat else []
        k = 0
        for d in batch:
            for _ in d:
                ids = enc[k].ids; k += 1
                stream.append(np.asarray(ids, np.uint16)); sent_off.append(sent_off[-1] + len(ids))
            doc_sent.append(doc_sent[-1] + len(d))
    toks = np.concatenate(stream) if stream else np.zeros(0, np.uint16)
    toks.tofile(P7_DATA / f"{s}_{role}_tokens.bin")
    np.save(P7_DATA / f"{s}_{role}_sent_off.npy", np.asarray(sent_off, np.int64))
    np.save(P7_DATA / f"{s}_{role}_doc_sent.npy", np.asarray(doc_sent, np.int64))
    so, ds = np.asarray(sent_off), np.asarray(doc_sent)
    lens = so[ds[1:]] - so[ds[:-1]]
    TOKSTATS[f"{s}_{role}"] = {"docs": len(lens), "tokens": int(lens.sum()), "median": float(np.median(lens)), "mean": float(lens.mean()),
                               "p90": float(np.percentile(lens, 90)), "max": int(lens.max()), "median_chunks512": float(np.median(np.ceil(lens / CHUNK_TOK)))}
t0 = time.time()
for s in SPLITS:
    save_tokens(s, "q", Q[s]["sents"]); save_tokens(s, "c", C[s]["sents"])
    save_json({"queries": Q[s]["ids"], "candidates": C[s]["ids"]}, P7_DATA / f"{s}_ids.json")
    if s in LABELLED: save_json({"gold": GOLD[s], "n_gold": N_GOLD[s]}, P7_DATA / f"{s}_gold.json")
    best = SYSTEMS[best_sys](s); top = topk(best, min(TOPN_SAVE, best.shape[1]))
    best_noyear = comb(s, W_PP) if best_sys.startswith("lexical++") else best
    save_json(CTX[s], P7_DATA / f"{s}_q_ctx.json")
    save_json({r: [y for y, _ in YEARS[s][r]] for r in ("q", "c")}, P7_DATA / f"{s}_years.json")
    np.savez_compressed(P7_DATA / f"{s}_lex.npz", best=best.astype(np.float32), top=top.astype(np.int32),
                        best_noyear=best_noyear.astype(np.float32), late=late(s, YEAR_TOL).astype(np.uint8),
                        prior=np.broadcast_to(PRIOR[s], best.shape).astype(np.float16) if s == "train" else PRIOR[s][0].astype(np.float16))
print(f"tokenised in {time.time()-t0:.0f}s")
display(pd.DataFrame(TOKSTATS).T.round(1))

pred = {}
top7 = topk(SYSTEMS[best_sys]("test"), K_EVAL)
for qi, q in enumerate(Q["test"]["ids"]):
    ids = [C["test"]["ids"][j] for j in top7[qi]]
    assert len(ids) == len(set(ids)) == min(K_EVAL, len(C["test"]["ids"])) and canon(q) not in {canon(x) for x in ids}
    pred[q] = ids
import zipfile
with zipfile.ZipFile(P7_BASE / "bm25_submission.zip", "w", zipfile.ZIP_DEFLATED) as zf: zf.writestr("prediction.json", json.dumps(pred, indent=1))
print("baseline submission:", P7_BASE / "bm25_submission.zip", "| queries:", len(pred), "| example:", next(iter(pred.items())))

meta = {"tokenizer": TOKENIZER, "chunk_tok": CHUNK_TOK, "k_eval": K_EVAL, "checks": CHK, "leak_checks": {k: str(v) for k, v in LEAK.items()},
        "best_ngram": best_n, "bm25_k1_b": BEST_KB, "fusion_weights": W_FUSE, "prior_beta": BETA, "best_lexical_system": best_sys,
        "lexpp_weights": {k: v for k, v in W_PP.items() if v}, "year_tol": YEAR_TOL, "year_gamma": YEAR_GAMMA if best_sys == "lexical++" else 0.0,
        "lexical_results": {f"{n} | {s}": r for (n, s), r in {**RES, **FINAL}.items()}, "tokens": TOKSTATS,
        "id_format_example": {"query": Q["train"]["ids"][0], "candidate": C["train"]["ids"][0]}}
save_json(meta, P7_DATA / "meta.json")
print(json.dumps({k: meta[k] for k in ("best_ngram", "bm25_k1_b", "fusion_weights", "prior_beta", "best_lexical_system", "lexpp_weights", "year_tol", "year_gamma")}, indent=1))
print("saved:", sorted(f"{p.name} ({p.stat().st_size/1e6:.1f} MB)" for p in P7_DATA.iterdir()))